# ETSI RAG — Quickstart Notebook

**Open Telco RAG**

This notebook walks through the Open Telco RAG pipeline for the **ETSI** corpus —
PDFs spanning 7 ISG groups (ENI, ZSM, MEC, NFV, SAI, F5G, RIS) plus
TC documents and ETSI Forge OpenAPI specifications.

By the end of this notebook you will have:
- Ingested ETSI documents into a local ChromaDB
- Queried ENI OODA model, ZSM automation, and MEC architecture
- Asked questions across all major ISG groups
- Evaluated retrieval quality with the evaluation metrics

---

### Prerequisites

```bash
git clone https://github.com/Znbne/telco-retrieve
cd open-telco-rag
pip install -e .
```

Download ETSI specifications:
```bash
python scripts/download_etsi.py --output data/etsi
```

Set your API keys in a `.env` file:
```
# For answer generation (choose one or both)
ANTHROPIC_API_KEY=sk-ant-...        # Claude models direct
OPENRO UTER_API_KEY=sk-or-...        # Any model via OpenRouter
```

## 1. Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from collections import Counter
import statistics

load_dotenv()

DATA_DIR   = Path("data/etsi")
DB_PATH    = "chroma_db"
MODEL_KEY  = "minilm"
COLLECTION = f"etsi_{MODEL_KEY}"

print(f"Data directory : {DATA_DIR}")
print(f"ChromaDB path  : {DB_PATH}")
print(f"Embedding model: {MODEL_KEY}")
print(f"Collection     : {COLLECTION}")

## 2. Check available documents

The ETSI corpus uses three naming conventions from different download sessions:
- **New-style**: `gs_ENI/`, `gr_ZSM/`, `gs_NFV_IFA/`
- **Old-style**: `ENI_gs/`, `ZSM_gr/`, `NFV_ts/`
- **ETSI Forge**: `mec.gs010-2-app-pkg-lcm-api`, `NFV-SOL006`

All three are handled by `ETSISource` via a folder map.

In [ ]:
if DATA_DIR.exists():
    folders = [d for d in DATA_DIR.iterdir() if d.is_dir()]
    pdfs    = list(DATA_DIR.rglob("*.pdf"))
    yamls   = list(DATA_DIR.rglob("*.yaml")) + list(DATA_DIR.rglob("*.yml"))
    print(f"Folders   : {len(folders)}")
    print(f"PDFs      : {len(pdfs):,}")
    print(f"YAML/APIs : {len(yamls)}")
    print("\nKey ISG folders confirmed:")
    isg_keywords = ["ENI", "ZSM", "MEC", "NFV", "SAI", "F5G", "RIS"]
    for kw in isg_keywords:
        matches = [d.name for d in folders if kw.lower() in d.name.lower()]
        status  = "✓ " + ", ".join(matches[:3]) if matches else "✗ not found"
        print(f"  {kw:<6} {status}")
else:
    print(f"[INFO] {DATA_DIR} not found.")
    print("Download: python scripts/download_etsi.py --output data/etsi")

## 3. Ingest into ChromaDB

The ETSI ingest pipeline:
1. Walks all folders using the `_FOLDER_MAP` to classify document type and ISG group
2. Loads PDFs and YAML/OpenAPI files (YAML converted to prose)
3. Uses a three-tier fallback for metadata: folder map → regex inference → filename
4. Applies paragraph-based chunking for PDFs, heading-aware for YAML

**Note**: JSON files are excluded to avoid duplicates with YAML renderings of
the same MEC/NFV specifications.

In [ ]:
from open_telco_rag.sources.etsi import ETSISource
from open_telco_rag.embedders.sentense_transformers import SentenceTransformerEmbedder
from open_telco_rag.stores.chroma import ChromaStore
from open_telco_rag.core.pipeline import ingest

source   = ETSISource()
embedder = SentenceTransformerEmbedder(model_key=MODEL_KEY)
store    = ChromaStore(path=DB_PATH, collection_name=COLLECTION)

print(f"Starting ingest — {store.count()} chunks already in collection")

if DATA_DIR.exists():
    stats = ingest(
        source     = source,
        store      = store,
        embedder   = embedder,
        data_dir   = str(DATA_DIR),
        chunk_size = 1024,
        overlap    = 200,
        resume     = True,
    )
    print(f"\nIngest complete")
    print(f"  Files processed : {stats.files_processed}")
    print(f"  Chunks stored   : {stats.chunks_stored}")
    print(f"  Skipped         : {stats.skipped_files}")
    print(f"  Errors          : {stats.error_files}")
print(f"  Total in DB     : {store.count():,}")

## 4. Explore the Collection

ETSI metadata includes ISG group (ENI, ZSM, MEC, NFV, SAI, F5G, RIS, TC)
and document type (GS, GR, TS, TR, EN, ES, SR).

In [ ]:
import chromadb

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(COLLECTION)
sample     = collection.get(limit=2000, include=["documents", "metadatas"])

group_counts = Counter(m.get("group", "?") for m in sample["metadatas"])
type_counts  = Counter(m.get("doc_type", "?") for m in sample["metadatas"])
lengths      = [len(d) for d in sample["documents"]]

ISG_DESC = {
    "ENI": "Experiential Networked Intelligence",
    "ZSM": "Zero-touch Service Management",
    "MEC": "Multi-access Edge Computing",
    "NFV": "Network Functions Virtualisation",
    "SAI": "Securing Artificial Intelligence",
    "F5G": "5th Generation Fixed Networks",
    "RIS": "Reconfigurable Intelligent Surfaces",
    "TC":  "Technical Committee documents",
}

print(f"Collection: {COLLECTION}")
print(f"Total chunks: {collection.count():,}\n")
print(f"ISG group distribution (sample of 2,000):")
print(f"{'Group':<8} {'Chunks':>8}  Description")
print("-" * 55)
for group, count in group_counts.most_common():
    desc = ISG_DESC.get(group, "")
    print(f"{group:<8} {count:>8}  {desc}")

print(f"\nDocument types: {dict(type_counts.most_common())}")
print(f"\nChunk length statistics (chars):")
print(f"  Mean   : {statistics.mean(lengths):.0f}")
print(f"  Median : {statistics.median(lengths):.0f}")

## 5. Semantic Search

ETSI ENI 005 defines the system architecture for cognitive network management
based on the OODA (Observe-Orient-Decide-Act) closed-loop model.

In [ ]:
from open_telco_rag.core.pipeline import search

QUERY = "What is the ETSI ENI OODA model and how does it support closed-loop automation?"
TOP_K = 5

results = search(
    QUERY,
    store    = store,
    embedder = embedder,
    top_k    = TOP_K,
)

print(f"Query: '{QUERY}'")
print(f"Top-{TOP_K} results:\n")
print(f"{'Rank':<5} {'Similarity':>10} {'Group':<8} {'Doc type':<8} {'Source'}")
print("-" * 75)

for i, r in enumerate(results, 1):
    sim  = round(1 - r.distance, 3)
    grp  = r.metadata.get("group", "")
    dt   = r.metadata.get("doc_type", "")
    src  = r.metadata.get("source", "")[:35]
    print(f"{i:<5} {sim:>10.3f} {grp:<8} {dt:<8} {src}")

print(f"\nTop result preview:")
print(results[0].text[:400])

### 5.1 ISG group coverage check

Run one query per ISG group and check coverage across the corpus.

In [ ]:
ISG_QUERIES = [
    ("ENI",  "What is the ENI OODA model for cognitive network management?"),
    ("ZSM",  "What does ETSI ZSM define for zero-touch service management?"),
    ("MEC",  "What are the ETSI MEC architecture reference points?"),
    ("NFV",  "How does ETSI NFV define the MANO architecture?"),
    ("SAI",  "How does ETSI SAI define security for AI systems?"),
    ("F5G",  "What is the ETSI F5G architecture for fifth generation fixed networks?"),
    ("RIS",  "What does ETSI RIS define for reconfigurable intelligent surfaces?"),
]

print(f"{'ISG':<6} {'Top group retrieved':<22} {'Mean sim':>10} {'Query'}")
print("-" * 80)

for expected_group, query in ISG_QUERIES:
    results  = search(query, store=store, embedder=embedder, top_k=5)
    sims     = [1 - r.distance for r in results]
    mean_sim = sum(sims) / len(sims) if sims else 0
    gc       = Counter(r.metadata.get("group", "?") for r in results)
    top_grp  = gc.most_common(1)[0][0] if gc else "?"
    hit      = "✓" if top_grp == expected_group else "✗"
    q_sh     = query[:40] + "..." if len(query) > 40 else query
    print(f"{expected_group:<6} {top_grp + ' ' + hit:<22} {mean_sim:>10.3f} {q_sh}")

## 6. Question Answering

In [ ]:
from open_telco_rag.core.pipeline import generate

QUESTION = "What is the ETSI ENI OODA model and how does it support closed-loop automation?"

retrieved = search(QUESTION, store=store, embedder=embedder, top_k=5)

from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

response  = generate(QUESTION, retrieved, source_type="etsi", generator=generator)

print(f"Question: {QUESTION}")
print(f"{'─'*70}")
print(f"\n{response['answer']}")
print(f"\n{'─'*70}")
print(f"Sources used: {response['sources']}")
print(f"Chunks used : {response['chunks_used']}")

### 6.1 Question bank

Try the full ETSI evaluation question set across all 7 ISG groups.

In [ ]:
QUESTIONS = [
    ("What is the ETSI ENI OODA model and how does it support closed-loop automation?", "ENI"),
    ("What does ETSI ZSM define for zero-touch service management?",                    "ZSM"),
    ("What are the ETSI MEC architecture and reference points?",                        "MEC"),
    ("How does ETSI NFV define the MANO architecture?",                                "NFV"),
    ("What are the ETSI ENI use cases for cognitive network management?",               "ENI"),
    ("How does ETSI SAI define security for AI systems?",                              "SAI"),
    ("What does the ETSI MEC application enablement API provide?",                     "MEC"),
    ("How does ETSI NFV-SOL define the VNF package format?",                           "NFV"),
    ("What is the ETSI F5G architecture for fifth generation fixed networks?",          "F5G"),
    ("What does ENI 005 specify for the ENI system architecture?",                     "ENI"),
]

print(f"{'#':<3} {'Expected ISG':<14} {'Top ISG retrieved':<19} {'Mean sim':>9} {'Question'}")
print("-" * 95)

for i, (question, expected_group) in enumerate(QUESTIONS, 1):
    results  = search(question, store=store, embedder=embedder, top_k=5)
    sims     = [1 - r.distance for r in results]
    mean_sim = sum(sims) / len(sims) if sims else 0
    gc       = Counter(r.metadata.get("group", "?") for r in results)
    top_grp  = gc.most_common(1)[0][0] if gc else "?"
    hit      = "✓" if top_grp == expected_group else "✗"
    q_short  = question[:42] + "..." if len(question) > 42 else question
    print(f"{i:<3} {expected_group:<14} {top_grp + ' ' + hit:<19} {mean_sim:>9.3f} {q_short}")

## 7. Evaluation

In [ ]:
from open_telco_rag.evaluation.evaluate import evaluate_corpus
from open_telco_rag.evaluation.questions import load_questions

questions = load_questions("etsi")
print(f"Loaded {len(questions)} evaluation questions for ETSI")
print(f"Questions with reference answers: {sum(1 for q in questions if q.reference_answer)}")

In [ ]:
from open_telco_rag.evaluation._claude import make_llm_call

# RAG generation model (answers questions)
rag_llm_call = make_llm_call("openrouter", "meta-llama/llama-3.3-70b-instruct")

# Metrics evaluation model (scores faithfulness, relevancy, etc.)
llm_call = make_llm_call("openrouter", "anthropic/claude-sonnet-4.5")

result = evaluate_corpus(
    corpus       = "etsi",
    store        = store,
    embedder     = embedder,
    questions    = questions,
    model        = MODEL_KEY,
    top_k        = 5,
    delay        = 1.0,
    llm_call     = llm_call,
    rag_llm_call = rag_llm_call,
)


In [ ]:
print(f"\nPer-question results — ETSI ({MODEL_KEY}, top-k=5)")
print(f"{'#':<3} {'Faith':>6} {'Relev':>6} {'Sim':>6} {'Recall':>7} {'Correct':>8} {'Question'}")
print("-" * 95)

for i, qr in enumerate(result.question_results, 1):
    q_short = qr.question[:45] + "..." if len(qr.question) > 45 else qr.question
    print(
        f"{i:<3} "
        f"{qr.faithfulness:>6.2f} "
        f"{qr.answer_relevancy:>6.2f} "
        f"{qr.mean_similarity:>6.3f} "
        f"{max(0, qr.context_recall):>7.2f} "
        f"{max(0, qr.answer_correctness):>8.2f} "
        f"{q_short}"
    )

print("-" * 95)
print(
    f"{'AVG':<3} "
    f"{result.mean_faithfulness:>6.2f} "
    f"{result.mean_answer_relevancy:>6.2f} "
    f"{result.mean_similarity:>6.3f} "
    f"{max(0, result.mean_context_recall):>7.2f} "
    f"{max(0, result.mean_answer_correctness):>8.2f}"
)


## 8. What the metrics tell us

| Metric | Type | Interpretation |
|--------|------|----------------|
| **Faithfulness** | Generation | Are the claims in the answer grounded in the retrieved chunks? |
| **Answer relevancy** | Generation | Does the answer address what was asked? |
| **Mean similarity** | Retrieval | How confident is the retrieval? Higher = better chunk matches |
| **Context recall** | Retrieval+GT | Were the chunks needed to answer the question retrieved? |
| **Answer correctness** | Generation+GT | Is the answer factually correct vs the reference answer? |
